In [3]:
import wandb
import pandas as pd
import numpy as np


group_names = [
    "CV-0-TransformerSymile-UKB-Fixed-Intersect-Proteomics-512",
    "CV-1-TransformerSymile-UKB-Fixed-Intersect-Proteomics-512",
    "CV-2-TransformerSymile-UKB-Fixed-Intersect-Proteomics-512",
    "CV-3-TransformerSymile-UKB-Fixed-Intersect-Proteomics-512",
    "CV-4-TransformerSymile-UKB-Fixed-Intersect-Proteomics-512",
]

entity = "cardiors"
project = "gatedsymile"
modelname = "TransformerSymile"

api = wandb.Api()

metric = "test/acc_top1"

rows = []
for group in group_names:
    # get runs in this group
    runs = api.runs(f"{entity}/{project}", {"group": group})

    # parse split/method from the group string
    parts = group.split("-")
    split = parts[1]          # "0", "1", "2"
    method = parts[2]         # "GatedSymile"
    split = f"CV-{split}"     # make it "CV-0" etc.

    for run in runs:
        summary = dict(run.summary)

        # put the metric into a column named exactly like `metric`
        rows.append({
            "id": run.id,
            "name": run.name,
            "group": run.group,
            "state": run.state,
            "method": method,
            "split": split,
            metric: summary.get(metric, np.nan),
        })

df = pd.DataFrame(rows)

# optional: keep only finished runs
# df = df[df["state"].isin(["finished"])].copy()

print("rows:", len(df))
print(df)
print(df[["split", "method", metric]].head())

metric = "test/acc_top1"

# 1) aggregate seeds within (method, split)
agg = (df.dropna(subset=[metric])
         .groupby(["method", "split"])[metric]
         .agg(["mean", "count"])
         .reset_index())

# optional sanity check: expect 3 seeds per cell
print(agg.pivot(index="split", columns="method", values="count"))

# 2) wide table: one row per split, columns=methods
wide = agg.pivot(index="split", columns="method", values="mean").sort_index()
display(wide)

# Method-wise mean +/- stderr
for m in [modelname]:
    x = wide[m].dropna().values.astype(float)
    K = x.size
    mean_x = x.mean()
    stderr_x = x.std(ddof=1) / np.sqrt(K)
    print(f"{m}: {mean_x:.6f} ± {stderr_x:.6f} (stderr, K={K})")

rows: 15
          id                    name  \
0   mor5udgx   twilight-elevator-520   
1   dqp3qvd9      woven-snowball-518   
2   4nf1hkpz          swept-star-518   
3   vx5kzdv0     super-resonance-524   
4   ssil6v7d     generous-donkey-517   
5   awwczxk2       hardy-firefly-516   
6   dp4zq3q1       dainty-meadow-527   
7   7y1xh6j5  sparkling-universe-526   
8   ajnhjm1g      usual-capybara-525   
9   y89e7o5y        warm-pyramid-530   
10  rfykpj8v          zesty-lake-529   
11  fv23xz1m          neat-blaze-528   
12  4iawb056         eager-shape-533   
13  a54gxlyi      eternal-yogurt-532   
14  x3u56wdg        kind-pyramid-531   

                                                group     state  \
0   CV-0-TransformerSymile-UKB-Fixed-Intersect-Pro...  finished   
1   CV-0-TransformerSymile-UKB-Fixed-Intersect-Pro...  finished   
2   CV-0-TransformerSymile-UKB-Fixed-Intersect-Pro...  finished   
3   CV-1-TransformerSymile-UKB-Fixed-Intersect-Pro...  finished   
4   CV-1-Transf

method,TransformerSymile
split,
CV-0,0.683757
CV-1,0.722765
CV-2,0.736762
CV-3,0.732313
CV-4,0.741102


TransformerSymile: 0.723340 ± 0.010352 (stderr, K=5)
